In [6]:
import pandas as pd
import numpy as np

In [7]:
dados_catalogo = {
"ID_Titulo": [101, 102, 103, 104, 105, 106, 101, 107, 108],
"Titulo": ["Stranger Things", "O Poderoso Chefão", "Interestelar",
"Breaking Bad", "Duna", "Succession", "Stranger Things", "Toy Story",
"Vingadores: Ultimato"],
"Tipo": ["Série", "Filme", "Filme", "Série", "Filme", "Série", "Série",
"Filme", "Filme"],
"Ano": [2016, 1972, np.nan, 2008, 2021, 2018, 2016, 1995, 2019],
"Genero": ["Ficção Científica", np.nan, "Ficção Científica", "Drama",
"Ficção Científica", "Drama", "Ficção Científica", "Animação", "Ação"],
"Plataforma": ["Netflix", "Prime Video", "Prime Video", "Netflix", "HBOMax", "HBO Max", "Netflix", "Disney+", "Disney+"]
}
df_catalogo = pd.DataFrame(dados_catalogo)

In [8]:
dados_metricas = {
"ID_Titulo": [101, 102, 103, 104, 105, 106, 107, 109], # O ID 109 é de um filme que não está no catálogo
"Nota_IMDb": [8.7, 9.2, 8.6, 9.5, 8.0, 8.8, 8.3, 9.0],
"Votos_Milhares": [1200, 1900, 1500, 1800, 700, 450, 600, 950],
"Orcamento_Milhoes": [12.0, 6.0, 165.0, 3.0, 165.0, 9.0, 30.0, 180.0]
}
df_metricas = pd.DataFrame(dados_metricas)
print("Tabelas carregadas com sucesso! Pronto para iniciar a análise.")


Tabelas carregadas com sucesso! Pronto para iniciar a análise.


In [ ]:
# Fase 1: O Trabalho Sujo
duplicadas = df_catalogo.duplicated().sum()
print(f"Linhas duplicadas encontradas: {duplicadas}")

df_catalogo = df_catalogo.drop_duplicates().copy()
df_catalogo['Genero'] = df_catalogo['Genero'].fillna('Cl?ssico')
df_catalogo.loc[df_catalogo['Titulo'] == 'Interestelar', 'Ano'] = 2014
df_catalogo['Ano'] = df_catalogo['Ano'].astype(int)

print('\nSoma de valores nulos por coluna:')
print(df_catalogo.isnull().sum())


In [ ]:
# Fase 2: Filtros e Enriquecimento de Dados
print('Titulo e Plataforma das 3 primeiras linhas:')
display(df_catalogo.loc[:2, ['Titulo', 'Plataforma']])

print('Titulo da quinta linha do catalogo:')
print(df_catalogo.loc[4, 'Titulo'])

filtro_fc = df_catalogo[(df_catalogo['Genero'] == 'Fic??o Cient?fica') & (df_catalogo['Ano'] >= 2015)]
print('\nProducoes de Fic??o Cient?fica a partir de 2015:')
display(filtro_fc)

def categoria_orcamento(valor):
    if valor < 10:
        return 'Baixo Custo'
    if valor < 100:
        return 'M?dio Custo'
    return 'Blockbuster'


In [ ]:
# Fase 3: Cruzamento e Agrupamentos
df_consolidado = df_catalogo.merge(df_metricas, on='ID_Titulo', how='left')
print('O ID_Titulo = 109 ficou de fora porque o merge left preserva todas as linhas do cat?logo e s? traz dados de m?tricas quando h? chave correspondente.')

df_consolidado['Nota_IMDb'] = df_consolidado['Nota_IMDb'].fillna(df_consolidado['Nota_IMDb'].mean())
df_consolidado['Votos_Milhares'] = df_consolidado['Votos_Milhares'].fillna(df_consolidado['Votos_Milhares'].mean())
df_consolidado['Orcamento_Milhoes'] = df_consolidado['Orcamento_Milhoes'].fillna(0.0)

resumo_genero = df_consolidado.groupby('Genero').agg(
    Orcamento_Milhoes=('Orcamento_Milhoes', 'sum'),
    Nota_IMDb=('Nota_IMDb', 'mean')
)

display(resumo_genero)

genero_maior_orcamento = resumo_genero['Orcamento_Milhoes'].idxmax()
genero_melhor_nota = resumo_genero['Nota_IMDb'].idxmax()

print(f"Genero com maior volume de or?amento total: {genero_maior_orcamento}")
print(f"Genero com melhor nota m?dia: {genero_melhor_nota}")

df_consolidado['Perfil_Financeiro'] = df_consolidado['Orcamento_Milhoes'].apply(categoria_orcamento)
display(df_consolidado)
